# KM6312 Team 7 - Part D Modelling

## Academic Failure Risk Classification and Academic Performance Regression

This notebook implements the two tasks assigned to Member D:

1. **Academic Failure Risk Prediction** (binary classification)
2. **Academic Performance Prediction** (regression)

The workflow follows the fixed train/test split produced by Member A. To prevent target leakage, the other outcome variables are excluded from each task. All transformations, class rebalancing, model fitting, cross-validation, and threshold selection are performed without using the held-out test set.

**Important interpretation rule:** the analysis estimates predictive associations in this dataset. It does not establish that AI use, social-media use, sleep, exercise, or health scores cause academic outcomes.

## 1. Environment and reproducibility

In [ ]:
from pathlib import Path
import json
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
import imblearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_recall_curve,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    root_mean_squared_error,
)
from sklearn.model_selection import (
    KFold,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_STATE = 42
N_JOBS = -1

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'axes.titlesize': 12,
    'legend.frameon': False,
})

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print('Python:', platform.python_version())
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('imbalanced-learn:', imblearn.__version__)
print('Project folder:', ROOT.name)

## 2. Load and validate Member A's hand-off

In [ ]:
raw = pd.read_csv(DATA_DIR / 'AI_SocialMedia_Student_Health_Dataset_clean.csv')
processed = pd.read_csv(DATA_DIR / 'processed_data.csv')

assert raw.shape == (15000, 14), f'Unexpected raw shape: {raw.shape}'
assert processed.shape == (15000, 16), f'Unexpected processed shape: {processed.shape}'
assert processed['split'].isin(['train', 'test']).all()
assert processed['split'].value_counts().to_dict() == {'train': 12000, 'test': 3000}
assert np.allclose(raw['Academic_Performance_Score'], processed['Academic_Performance_Score'])
assert raw['Academic_Failure_Risk'].equals(processed['Academic_Failure_Risk'])

burnout_map = {'Low': 0, 'Moderate': 1, 'High': 2, 'Severe': 3}
assert raw['Burnout_Level'].map(burnout_map).equals(processed['Burnout_Level'])

data = raw.drop(columns='Student_ID').copy()
data['split'] = processed['split'].to_numpy()

print('Data shape:', data.shape)
print()
print('Split counts:')
print(data['split'].value_counts())
print()
print('Failure-risk counts by split:')
print(pd.crosstab(data['split'], data['Academic_Failure_Risk']))
print()
print('Academic-performance summary by split:')
display(data.groupby('split')['Academic_Performance_Score'].agg(['count', 'mean', 'std', 'min', 'max']).round(3))

### Leakage controls

The three outcomes are strongly related and must not be used to predict one another:

- Regression target: `Academic_Performance_Score`  
  Exclude `Burnout_Level` and `Academic_Failure_Risk`.
- Binary-classification target: `Academic_Failure_Risk`  
  Exclude `Burnout_Level` and `Academic_Performance_Score`.

The model inputs are restricted to recorded demographic, behavioural, and health variables.

In [ ]:
continuous_features = [
    'Age',
    'Daily_Social_Media_Hours',
    'Daily_AI_Tool_Usage_Hours',
    'Sleep_Hours',
    'Physical_Activity_Hours',
    'Mental_Health_Score',
    'Physical_Health_Score',
    'Social_Isolation_Score',
]
categorical_features = ['Gender', 'Education_Level']
predictor_features = continuous_features + categorical_features

forbidden = {'Student_ID', 'Burnout_Level', 'Academic_Performance_Score', 'Academic_Failure_Risk', 'split'}
assert forbidden.isdisjoint(predictor_features)

train_mask = data['split'].eq('train')
test_mask = data['split'].eq('test')

X_train = data.loc[train_mask, predictor_features].copy()
X_test = data.loc[test_mask, predictor_features].copy()

y_train_reg = data.loc[train_mask, 'Academic_Performance_Score'].copy()
y_test_reg = data.loc[test_mask, 'Academic_Performance_Score'].copy()
y_train_cls = data.loc[train_mask, 'Academic_Failure_Risk'].astype(int).copy()
y_test_cls = data.loc[test_mask, 'Academic_Failure_Risk'].astype(int).copy()

print('Predictors:', predictor_features)
print('Regression train/test:', X_train.shape, X_test.shape)
print('Classification positive prevalence:')
print('  train =', round(y_train_cls.mean(), 4))
print('  test  =', round(y_test_cls.mean(), 4))

## 3. Shared preprocessing pipelines

In [ ]:
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), continuous_features),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
        ]), categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', SimpleImputer(strategy='median'), continuous_features),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
        ]), categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

# For SMOTENC, numeric variables are scaled before neighbour calculation and
# categorical variables are ordinal-encoded. After resampling, categorical
# variables are one-hot encoded before model fitting.
pre_smote = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), continuous_features),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), categorical_features),
    ],
    remainder='drop',
    sparse_threshold=0,
)

categorical_indices_after_pre_smote = list(
    range(len(continuous_features), len(continuous_features) + len(categorical_features))
)

post_smote = ColumnTransformer(
    transformers=[
        ('numeric', 'passthrough', list(range(len(continuous_features)))),
        ('categorical', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False),
         categorical_indices_after_pre_smote),
    ],
    remainder='drop',
    sparse_threshold=0,
)

print('Continuous features:', len(continuous_features))
print('Categorical features:', len(categorical_features))

# Part I - Academic Performance Regression

## 4. Regression model specification

Three models are compared:

- Mean prediction (`DummyRegressor`) as a non-informative baseline
- Multiple linear regression for adjusted directional associations
- Random-forest regression for non-linear patterns and interactions

Random-forest hyperparameters are selected using five-fold cross-validation on the training set only. The held-out test set is evaluated once after model selection.

In [ ]:
regression_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

regression_models = {
    'Mean Baseline': Pipeline([
        ('preprocess', scaled_preprocessor),
        ('model', DummyRegressor(strategy='mean')),
    ]),
    'Linear Regression': Pipeline([
        ('preprocess', scaled_preprocessor),
        ('model', LinearRegression()),
    ]),
}

rf_regression_pipeline = Pipeline([
    ('preprocess', tree_preprocessor),
    ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)),
])

rf_regression_search_space = {
    'model__n_estimators': [250, 400, 600],
    'model__max_depth': [None, 8, 12, 16],
    'model__min_samples_leaf': [1, 2, 4, 8],
    'model__max_features': ['sqrt', 0.7, 1.0],
}

rf_regression_search = RandomizedSearchCV(
    estimator=rf_regression_pipeline,
    param_distributions=rf_regression_search_space,
    n_iter=12,
    scoring='neg_root_mean_squared_error',
    cv=regression_cv,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    refit=True,
    return_train_score=False,
    verbose=0,
)

rf_regression_search.fit(X_train, y_train_reg)
regression_models['Random Forest'] = rf_regression_search.best_estimator_

print('Best random-forest parameters:')
print(rf_regression_search.best_params_)
print('Best training CV RMSE:', round(-rf_regression_search.best_score_, 4))

with open(TABLE_DIR / 'regression_best_parameters.json', 'w', encoding='utf-8') as f:
    json.dump(rf_regression_search.best_params_, f, indent=2)

## 5. Regression cross-validation and held-out test results

In [ ]:
regression_scoring = {
    'RMSE': 'neg_root_mean_squared_error',
    'MAE': 'neg_mean_absolute_error',
    'R2': 'r2',
}

regression_cv_rows = []
regression_test_rows = []
regression_fitted_models = {}
regression_predictions = {}

for name, model in regression_models.items():
    cv_result = cross_validate(
        model,
        X_train,
        y_train_reg,
        scoring=regression_scoring,
        cv=regression_cv,
        n_jobs=N_JOBS,
        return_train_score=False,
    )
    regression_cv_rows.append({
        'Model': name,
        'CV RMSE Mean': -cv_result['test_RMSE'].mean(),
        'CV RMSE SD': cv_result['test_RMSE'].std(),
        'CV MAE Mean': -cv_result['test_MAE'].mean(),
        'CV R2 Mean': cv_result['test_R2'].mean(),
        'CV R2 SD': cv_result['test_R2'].std(),
    })

    fitted = clone(model).fit(X_train, y_train_reg)
    predictions = fitted.predict(X_test)
    regression_fitted_models[name] = fitted
    regression_predictions[name] = predictions
    regression_test_rows.append({
        'Model': name,
        'RMSE': root_mean_squared_error(y_test_reg, predictions),
        'MAE': mean_absolute_error(y_test_reg, predictions),
        'R2': r2_score(y_test_reg, predictions),
    })

regression_cv_table = pd.DataFrame(regression_cv_rows).sort_values('CV RMSE Mean')
regression_test_table = pd.DataFrame(regression_test_rows).sort_values('RMSE')

regression_cv_table.to_csv(TABLE_DIR / 'regression_cross_validation.csv', index=False)
regression_test_table.to_csv(TABLE_DIR / 'regression_test_metrics.csv', index=False)

print('Five-fold cross-validation results:')
display(regression_cv_table.round(4))
print('Held-out test results:')
display(regression_test_table.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
order = regression_test_table['Model'].tolist()

sns.barplot(data=regression_test_table, x='RMSE', y='Model', order=order, color='#4C78A8', ax=axes[0])
axes[0].set_title('Regression Test RMSE (lower is better)')
axes[0].set_xlabel('RMSE')
axes[0].set_ylabel('')

sns.barplot(data=regression_test_table, x='R2', y='Model', order=order, color='#F58518', ax=axes[1])
axes[1].set_title('Regression Test R² (higher is better)')
axes[1].set_xlabel('R²')
axes[1].set_ylabel('')

fig.tight_layout()
fig.savefig(FIGURE_DIR / 'regression_model_comparison.png', bbox_inches='tight')
plt.show()

## 6. Regression diagnostics and interpretation

In [ ]:
best_regression_name = regression_test_table.iloc[0]['Model']
best_regression_predictions = regression_predictions[best_regression_name]
regression_residuals = y_test_reg.to_numpy() - best_regression_predictions

fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))

axes[0].scatter(y_test_reg, best_regression_predictions, alpha=0.35, s=16, color='#4C78A8')
low = min(y_test_reg.min(), best_regression_predictions.min())
high = max(y_test_reg.max(), best_regression_predictions.max())
axes[0].plot([low, high], [low, high], '--', color='black', linewidth=1)
axes[0].set_title(f'Observed vs Predicted ({best_regression_name})')
axes[0].set_xlabel('Observed academic performance')
axes[0].set_ylabel('Predicted academic performance')

axes[1].scatter(best_regression_predictions, regression_residuals, alpha=0.35, s=16, color='#E45756')
axes[1].axhline(0, linestyle='--', color='black', linewidth=1)
axes[1].set_title('Residual Plot')
axes[1].set_xlabel('Predicted academic performance')
axes[1].set_ylabel('Residual (observed - predicted)')

fig.tight_layout()
fig.savefig(FIGURE_DIR / 'regression_diagnostics.png', bbox_inches='tight')
plt.show()

print('Best held-out regression model:', best_regression_name)

In [ ]:
linear_model = regression_fitted_models['Linear Regression']
linear_feature_names = linear_model.named_steps['preprocess'].get_feature_names_out()
linear_coefficients = pd.DataFrame({
    'Feature': linear_feature_names,
    'Coefficient': linear_model.named_steps['model'].coef_,
}).sort_values('Coefficient')
linear_coefficients['Absolute Coefficient'] = linear_coefficients['Coefficient'].abs()
linear_coefficients.to_csv(TABLE_DIR / 'linear_regression_coefficients.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5.2))
colors = np.where(linear_coefficients['Coefficient'] >= 0, '#4C78A8', '#E45756')
ax.barh(linear_coefficients['Feature'], linear_coefficients['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Linear Regression Coefficients')
ax.set_xlabel('Coefficient (numeric variables standardized; categoricals relative to reference)')
ax.set_ylabel('')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'linear_regression_coefficients.png', bbox_inches='tight')
plt.show()

display(linear_coefficients.sort_values('Absolute Coefficient', ascending=False).round(4))

In [ ]:
rf_regression_model = regression_fitted_models['Random Forest']
permutation = permutation_importance(
    rf_regression_model,
    X_test,
    y_test_reg,
    scoring='neg_root_mean_squared_error',
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)

regression_importance = pd.DataFrame({
    'Feature': predictor_features,
    'Importance Mean': permutation.importances_mean,
    'Importance SD': permutation.importances_std,
}).sort_values('Importance Mean', ascending=False)
regression_importance.to_csv(TABLE_DIR / 'regression_permutation_importance.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 5.0))
plot_data = regression_importance.sort_values('Importance Mean')
ax.barh(
    plot_data['Feature'],
    plot_data['Importance Mean'],
    xerr=plot_data['Importance SD'],
    color='#54A24B',
    alpha=0.9,
)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Random-Forest Permutation Importance')
ax.set_xlabel('Increase in test RMSE after permutation')
ax.set_ylabel('')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'regression_permutation_importance.png', bbox_inches='tight')
plt.show()

display(regression_importance.round(4))

### Interpretation note

Linear-regression coefficients describe adjusted associations, not causal effects. Numeric coefficients are expressed per one training-set standard deviation. Random-forest permutation importance measures predictive contribution on the held-out test set, but it does not show whether an association is positive or negative.

# Part II - Academic Failure Risk Classification

## 7. Classification model specification

The positive class represents students labelled as being at risk of academic failure. Because only 6% of records are positive, accuracy is not a sufficient evaluation measure. The primary model-selection metric is **PR-AUC (average precision)**, supported by ROC-AUC, precision, recall, F1, balanced accuracy, and confusion matrices.

Four model variants are compared:

1. Logistic regression with balanced class weights
2. Random forest with balanced class weights
3. Logistic regression with SMOTENC oversampling
4. Random forest with SMOTENC oversampling

SMOTENC is used instead of ordinary SMOTE because the predictors contain categorical variables.

In [ ]:
classification_models = {
    'Logistic (class weight)': Pipeline([
        ('preprocess', scaled_preprocessor),
        ('model', LogisticRegression(
            class_weight='balanced',
            max_iter=3000,
            solver='lbfgs',
            random_state=RANDOM_STATE,
        )),
    ]),
    'Random Forest (class weight)': Pipeline([
        ('preprocess', tree_preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=500,
            max_depth=12,
            min_samples_leaf=3,
            max_features='sqrt',
            class_weight='balanced_subsample',
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),
    'Logistic (SMOTENC)': ImbPipeline([
        ('pre_smote', pre_smote),
        ('smote', SMOTENC(
            categorical_features=categorical_indices_after_pre_smote,
            random_state=RANDOM_STATE,
        )),
        ('post_smote', post_smote),
        ('model', LogisticRegression(
            max_iter=3000,
            solver='lbfgs',
            random_state=RANDOM_STATE,
        )),
    ]),
    'Random Forest (SMOTENC)': ImbPipeline([
        ('pre_smote', pre_smote),
        ('smote', SMOTENC(
            categorical_features=categorical_indices_after_pre_smote,
            random_state=RANDOM_STATE,
        )),
        ('post_smote', post_smote),
        ('model', RandomForestClassifier(
            n_estimators=500,
            max_depth=12,
            min_samples_leaf=3,
            max_features='sqrt',
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),
}

classification_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print('Classification models:', list(classification_models))

## 8. Classification cross-validation

In [ ]:
classification_scoring = {
    'PR_AUC': 'average_precision',
    'ROC_AUC': 'roc_auc',
    'F1': 'f1',
    'Recall': 'recall',
    'Precision': 'precision',
    'Balanced_Accuracy': 'balanced_accuracy',
}

classification_cv_rows = []

for name, model in classification_models.items():
    cv_result = cross_validate(
        model,
        X_train,
        y_train_cls,
        scoring=classification_scoring,
        cv=classification_cv,
        n_jobs=N_JOBS,
        return_train_score=False,
    )
    classification_cv_rows.append({
        'Model': name,
        'CV PR-AUC Mean': cv_result['test_PR_AUC'].mean(),
        'CV PR-AUC SD': cv_result['test_PR_AUC'].std(),
        'CV ROC-AUC Mean': cv_result['test_ROC_AUC'].mean(),
        'CV F1 Mean': cv_result['test_F1'].mean(),
        'CV Recall Mean': cv_result['test_Recall'].mean(),
        'CV Precision Mean': cv_result['test_Precision'].mean(),
        'CV Balanced Accuracy Mean': cv_result['test_Balanced_Accuracy'].mean(),
    })

classification_cv_table = pd.DataFrame(classification_cv_rows).sort_values(
    'CV PR-AUC Mean', ascending=False
)
classification_cv_table.to_csv(TABLE_DIR / 'classification_cross_validation.csv', index=False)
display(classification_cv_table.round(4))

## 9. Held-out classification results at the default threshold

In [ ]:
def classification_metrics(y_true, probabilities, threshold=0.5):
    predictions = (probabilities >= threshold).astype(int)
    return {
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_true, predictions),
        'Balanced Accuracy': balanced_accuracy_score(y_true, predictions),
        'Precision': precision_score(y_true, predictions, zero_division=0),
        'Recall': recall_score(y_true, predictions, zero_division=0),
        'F1': f1_score(y_true, predictions, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, probabilities),
        'PR-AUC': average_precision_score(y_true, probabilities),
    }

classification_fitted_models = {}
classification_probabilities = {}
classification_test_rows = []

for name, model in classification_models.items():
    fitted = clone(model).fit(X_train, y_train_cls)
    probabilities = fitted.predict_proba(X_test)[:, 1]
    classification_fitted_models[name] = fitted
    classification_probabilities[name] = probabilities
    row = {'Model': name, **classification_metrics(y_test_cls, probabilities, threshold=0.5)}
    classification_test_rows.append(row)

classification_test_table = pd.DataFrame(classification_test_rows).sort_values(
    'PR-AUC', ascending=False
)
classification_test_table.to_csv(TABLE_DIR / 'classification_test_metrics_default_threshold.csv', index=False)
display(classification_test_table.round(4))

In [ ]:
metrics_to_plot = ['PR-AUC', 'F1', 'Recall', 'Precision']
plot_table = classification_test_table.set_index('Model')[metrics_to_plot]

ax = plot_table.plot(kind='bar', figsize=(10, 4.8), width=0.8, colormap='viridis')
ax.set_title('Classification Performance on the Held-out Test Set')
ax.set_ylabel('Metric value')
ax.set_xlabel('')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=18)
ax.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, 1.18))
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'classification_model_comparison.png', bbox_inches='tight')
plt.show()

## 10. Threshold selection using out-of-fold training predictions

In [ ]:
best_classification_name = classification_cv_table.iloc[0]['Model']
best_classification_model = classification_models[best_classification_name]

oof_probabilities = cross_val_predict(
    best_classification_model,
    X_train,
    y_train_cls,
    cv=classification_cv,
    method='predict_proba',
    n_jobs=N_JOBS,
)[:, 1]

threshold_grid = np.linspace(0.02, 0.98, 193)
threshold_rows = []
for threshold in threshold_grid:
    predictions = (oof_probabilities >= threshold).astype(int)
    threshold_rows.append({
        'Threshold': threshold,
        'Precision': precision_score(y_train_cls, predictions, zero_division=0),
        'Recall': recall_score(y_train_cls, predictions, zero_division=0),
        'F1': f1_score(y_train_cls, predictions, zero_division=0),
    })

threshold_table = pd.DataFrame(threshold_rows)
best_threshold = float(threshold_table.loc[threshold_table['F1'].idxmax(), 'Threshold'])
threshold_table.to_csv(TABLE_DIR / 'classification_threshold_search.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
for metric, color in [('Precision', '#4C78A8'), ('Recall', '#E45756'), ('F1', '#54A24B')]:
    ax.plot(threshold_table['Threshold'], threshold_table[metric], label=metric, color=color)
ax.axvline(best_threshold, linestyle='--', color='black', label=f'Chosen threshold = {best_threshold:.3f}')
ax.set_title(f'Threshold Selection from Training OOF Predictions\n{best_classification_name}')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Metric value')
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'classification_threshold_selection.png', bbox_inches='tight')
plt.show()

print('Best classification model by CV PR-AUC:', best_classification_name)
print('F1-optimal threshold selected from training OOF predictions:', round(best_threshold, 4))

## 11. Final classification evaluation and curves

In [ ]:
best_test_probabilities = classification_probabilities[best_classification_name]
final_classification_metrics = pd.DataFrame([
    {
        'Model': best_classification_name,
        'Threshold Rule': 'Default 0.5',
        **classification_metrics(y_test_cls, best_test_probabilities, threshold=0.5),
    },
    {
        'Model': best_classification_name,
        'Threshold Rule': 'Training OOF F1-optimal',
        **classification_metrics(y_test_cls, best_test_probabilities, threshold=best_threshold),
    },
])
final_classification_metrics.to_csv(
    TABLE_DIR / 'classification_best_model_threshold_comparison.csv', index=False
)
display(final_classification_metrics.round(4))

final_predictions = (best_test_probabilities >= best_threshold).astype(int)
cm_counts = confusion_matrix(y_test_cls, final_predictions)
cm_normalized = confusion_matrix(y_test_cls, final_predictions, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
sns.heatmap(cm_counts, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0])
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted class')
axes[0].set_ylabel('True class')

sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', cbar=False, ax=axes[1])
axes[1].set_title('Confusion Matrix (Row-normalized)')
axes[1].set_xlabel('Predicted class')
axes[1].set_ylabel('True class')

fig.suptitle(f'{best_classification_name} at threshold {best_threshold:.3f}', y=1.03, fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'classification_confusion_matrix.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))

for name, probabilities in classification_probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test_cls, probabilities)
    pr_auc = average_precision_score(y_test_cls, probabilities)
    axes[0].plot(recall, precision, label=f'{name} ({pr_auc:.3f})')

axes[0].axhline(y_test_cls.mean(), linestyle='--', color='grey', label=f'Prevalence ({y_test_cls.mean():.2f})')
axes[0].set_title('Precision-Recall Curves')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1)

for name, probabilities in classification_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test_cls, probabilities)
    roc_auc = roc_auc_score(y_test_cls, probabilities)
    axes[1].plot(fpr, tpr, label=f'{name} ({roc_auc:.3f})')

axes[1].plot([0, 1], [0, 1], '--', color='grey')
axes[1].set_title('ROC Curves')
axes[1].set_xlabel('False-positive rate')
axes[1].set_ylabel('True-positive rate')
axes[1].set_xlim(0, 1)
axes[1].set_ylim(0, 1)

axes[0].legend(loc='lower left', fontsize=8, title='PR-AUC')
axes[1].legend(loc='lower right', fontsize=8, title='ROC-AUC')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'classification_pr_roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Permutation importance is calculated after model selection for interpretation only.
# It is not used to select a model or tune the decision threshold.
classification_importance = permutation_importance(
    classification_fitted_models[best_classification_name],
    X_test,
    y_test_cls,
    scoring='average_precision',
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
)

classification_importance_table = pd.DataFrame({
    'Feature': predictor_features,
    'Importance Mean': classification_importance.importances_mean,
    'Importance SD': classification_importance.importances_std,
}).sort_values('Importance Mean', ascending=False)
classification_importance_table.to_csv(
    TABLE_DIR / 'classification_permutation_importance.csv', index=False
)

plot_data = classification_importance_table.sort_values('Importance Mean')
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    plot_data['Feature'],
    plot_data['Importance Mean'],
    xerr=plot_data['Importance SD'],
    color='#B279A2',
    alpha=0.9,
)
ax.axvline(0, color='black', linewidth=1)
ax.set_title(f'{best_classification_name}: Permutation Importance')
ax.set_xlabel('Decrease in test PR-AUC after permutation')
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'classification_permutation_importance.png', bbox_inches='tight')
plt.show()

display(classification_importance_table.round(4))

### Classification interpretation note

The positive class is rare, so PR-AUC and recall are more informative than raw accuracy. The tuned decision threshold is selected exclusively from out-of-fold training predictions; the test set is not used to choose the threshold. The observed relationship between failure risk and the available predictors is predictive rather than causal, and the label-generation process remains a dataset limitation.

## 12. Reproducibility summary and report-ready outputs

In [ ]:
summary = {
    'random_state': RANDOM_STATE,
    'train_rows': int(train_mask.sum()),
    'test_rows': int(test_mask.sum()),
    'classification_positive_train': int(y_train_cls.sum()),
    'classification_positive_test': int(y_test_cls.sum()),
    'best_regression_model_by_test_rmse': best_regression_name,
    'best_classification_model_by_cv_pr_auc': best_classification_name,
    'classification_threshold_selected_from_training_oof': best_threshold,
}

with open(TABLE_DIR / 'analysis_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print()
print('Saved tables:')
for path in sorted(TABLE_DIR.glob('*')):
    print('-', path.name)
print()
print('Saved figures:')
for path in sorted(FIGURE_DIR.glob('*.png')):
    print('-', path.name)

## 13. Conclusions for Part D

Use the executed metric tables above to write the final report. Do not copy the proposal's expected scores as if they were actual findings. Report the held-out test results, explain the gap between expected and observed predictive performance, and state the following limitations explicitly:

- the data source and label-generation procedure limit external validity;
- target labels exhibit strong dependencies and must not be used as predictors of one another;
- prediction does not establish causality;
- Age and Education Level contain overlapping information;
- the minority positive class makes performance sensitive to the decision threshold.

The final report should prioritize a small number of clear tables and figures rather than include every diagnostic generated by this notebook.